In [2]:
!pip install seqeval
!pip install -q "protobuf<5" "transformers>=4.35.0"

In [3]:
# libraries for data loading and system/path settings
import json
import sys
import warnings
import os
import random
import matplotlib.pyplot as plt
import numpy as np

# libraries for model building and training
from seqeval.metrics import classification_report as seqeval_classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from transformers import AutoTokenizer, AutoModelForTokenClassification, logging
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

# global settings to suppress unproblematic warning messages
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="The sentencepiece tokenizer")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" 

In [4]:
# load the augmented data in json format
with open("/kaggle/input/training-validation-ner/training_set.json", "r") as f:
    data_augmented = json.load(f)

# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in data_augmented:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

In [6]:
def tokenization_labelling(text, entities, tokenizer, tag2id, max_len):

    # get the encoding of the sentence
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True,
                         max_length=max_len, padding="max_length")
    
    # create preliminary list with O tags for all tokens
    tags = ["O"] * len(encoding.offset_mapping)

    # loop over annotations and extract start and end index as well as the given tag
    for ent in entities:
        start, end = ent["start"], ent["end"]
        ent_tag = ent["tag"][0:2]

        # loop over all tokens in the sentence and check for overlap
        for idx, (token_start, token_end) in enumerate(encoding.offset_mapping):
            # continue if it is a special token
            if token_start == token_end == 0:
                continue
            # check for overlap (overlap checking strictly necessary for deberta)
            if token_end > start and token_start < end:
                # assign B-tag if start is equal or smaller (smaller if deberta)
                if token_start <= start:
                    tags[idx] = f"B-{ent_tag}"
                # otherwise it is an inside token
                else:
                    tags[idx] = f"I-{ent_tag}"

    # extract the word ids
    word_ids = encoding.word_ids()

    # ensure propagate B-tags are propagated to all subwords of the same word (only actually relevant for deberta)
    for idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        # if token has B-tag ensure that all other tokens of the same word get I-tag
        if tags[idx].startswith("B-"):
            for j, wid2 in enumerate(word_ids):
                if wid2 == wid and j != idx:
                    tags[j] = f"I-{ent_tag}"

    # convert tags to IDs, masking special tokens
    tag_ids = [-100 if wid is None else tag2id.get(tag, tag2id["O"])
               for tag, wid in zip(tags, encoding.word_ids())]

    return encoding["input_ids"], encoding["attention_mask"], tag_ids, word_ids


def run_testset_ner(model, test_dataloader, id2tag, device, for_metric):
    
    model.eval()

    all_true_tags, all_pred_tags = [], []
    all_true_spans, all_pred_spans = [], []
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            tag_ids = batch["tag_ids"].to(device)
            batch_word_ids = batch["word_ids"]

            outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=tag_ids)
            logits = outputs.logits
            loss = outputs.loss
            total_loss += loss.item()
            num_batches += 1
            predictions = torch.argmax(logits, dim=2)

            if for_metric == "seqeval":
                for i in range(len(tag_ids)):
                    true_seq = tag_ids[i].cpu().numpy()
                    pred_seq = predictions[i].cpu().numpy()

                    true_tags = []
                    pred_tags = []
                    for t, p in zip(true_seq, pred_seq):
                        if t != -100:
                            true_tags.append(id2tag[t])
                            pred_tags.append(id2tag[p])

                    all_true_tags.append(true_tags)
                    all_pred_tags.append(pred_tags)

            elif for_metric == "cross_span":
                for i in range(len(tag_ids)):
                    true_seq = tag_ids[i].cpu().numpy()
                    pred_seq = predictions[i].cpu().numpy()
                    word_ids = batch_word_ids[i]

                    word_level_tags, _ = __labels_to_wordlevel_tags(true_seq, id2tag, word_ids)
                    all_true_spans.append(extract_spans(word_level_tags))

                    word_level_tags, _ = __labels_to_wordlevel_tags(pred_seq, id2tag, word_ids)
                    all_pred_spans.append(extract_spans(word_level_tags))

            elif for_metric == "sentence_level":
                for i in range(len(tag_ids)):
                    true_seq = tag_ids[i].cpu().numpy()
                    pred_seq = predictions[i].cpu().numpy()
                    word_ids = batch_word_ids[i]

                    true_word_tags, _ = __labels_to_wordlevel_tags(true_seq, id2tag, word_ids)
                    pred_word_tags, _ = __labels_to_wordlevel_tags(pred_seq, id2tag, word_ids)

                    all_true_tags.append(true_word_tags)
                    all_pred_tags.append(pred_word_tags)

    avg_loss = total_loss / num_batches

    if for_metric == "seqeval":
        return all_true_tags, all_pred_tags, avg_loss
    elif for_metric == "cross_span":
        return all_true_spans, all_pred_spans, avg_loss
    elif for_metric == "sentence_level":
        return all_true_tags, all_pred_tags, avg_loss

def evaluate_seqeval(all_true_tags, all_pred_tags):
    classification_report = seqeval_classification_report(all_true_tags, all_pred_tags, output_dict=True)
    precision = classification_report["sg"]["precision"]
    recall = classification_report["sg"]["recall"]
    f1_score = classification_report["sg"]["f1-score"]

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1_score
        }

In [7]:
class TokenDataset(Dataset):
    def __init__(self, data, tokenizer, tag2id, max_len):
        self.dataset = []
        self.max_len = max_len

        for task in data:
            # get the sentence and all annotations
            text = task["sentence"]
            spans = task["annotations"]

            # tokenize and get all ids
            input_ids, attention_mask, tag_ids, word_ids = tokenization_labelling(text, spans, tokenizer, tag2id, self.max_len)

            # add everything to the dataset list
            self.dataset.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "tag_ids": torch.tensor(tag_ids, dtype=torch.long),
                "word_ids": word_ids})

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

def custom_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_masks = torch.stack([item["attention_mask"] for item in batch])
    tag_ids = torch.stack([item["tag_ids"] for item in batch])
    word_ids = [item["word_ids"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "tag_ids": tag_ids,
        "word_ids": word_ids
    }

## Evaluation of General Dataset Size

In [10]:
# set a seed to ensure reproducibility
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# set different dataset sizes to test
data_sizes = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]
num_folds = 5
metrics_non_augmented = []

# set model name and specific hyperparameters to test
model_name = "microsoft/deberta-v3-base" #"roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 2e-05
weight_decay = 0.3
batch_size = 8
epochs = 11
tokenizer = AutoTokenizer.from_pretrained(model_name)

# loop through different data sizes
for size in data_sizes:

    print(f"Starting with size {size}")

    # subset the data to size
    #data_subset = random.sample(data_augmented, size)

    # create list to store fold metrics in
    fold_metrics_non_augmented = []

    # create K-Fold splits
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)

    for fold, (train_idx, val_idx) in enumerate(kf.split(data_augmented)):
        
        # split the data
        train_fold_data = [data_augmented[i] for i in train_idx]
        val_fold_data = [data_augmented[i] for i in val_idx]

        # create datasets and dataloaders
        train_dataset = TokenDataset(train_fold_data, tokenizer, tag_to_id, max_len=128)
        val_dataset = TokenDataset(val_fold_data, tokenizer, tag_to_id, max_len=128)
        
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

        # create model and optimizer for the non-augmentations
        model_non_augmented = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(tag_to_id),
            id2label=id_to_tag,
            label2id=tag_to_id
        ).to(device)
        optimizer_non_aug = AdamW(model_non_augmented.parameters(), lr=lr, weight_decay=weight_decay)

        # create early stopping object
        early_stopper_non_aug = EarlyStopping(patience=5, min_delta=0.0001, save_model=False, path='checkpoint.pt', printoption=False)

        # loop through the epochs
        for epoch in range(epochs):
            model_non_augmented.train()
            total_loss = 0.0
            for batch in train_dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                tag_ids = batch["tag_ids"].to(device)

                optimizer_non_aug.zero_grad()

                with torch.autocast(device_type=device.type, dtype=torch.float16):
                    outputs = model_non_augmented(input_ids=input_ids, attention_mask=attention_mask, labels=tag_ids)
                    loss = outputs.loss

                total_loss += loss.item()
                loss.backward()
                clip_grad_norm_(model_non_augmented.parameters(), 1.0)
                optimizer_non_aug.step()

            # calculate validation performance and feed into early stopper
            all_true, all_pred, _ = run_testset_ner(
                model=model_non_augmented, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
            )
            metrics_seqeval = evaluate_seqeval(all_true, all_pred)
            val_f1 = metrics_seqeval["f1"]
            print(f"Current validation f1-score: {val_f1}")
            early_stopper_non_aug(val_f1, model_non_augmented, epoch)
            if early_stopper_non_aug.early_stop:
                print("Early stopping triggered.")
                break    
                
        # get the best f1 score
        best_f1 = early_stopper_non_aug.best_f1
        print(f"Best f1-score of the fold: {best_f1}")
        fold_metrics_non_augmented.append(best_f1)

    # take averages across folds
    mean_seqeval_non_augmented = np.mean(fold_metrics_non_augmented)

    # take standard deviation
    sd_seqeval_non_augmented = np.std(fold_metrics_non_augmented)

    # calculate confidence intervals
    ci_non_aug = 1.96 * sd_seqeval_non_augmented / np.sqrt(5)

    metrics_non_augmented.append(
        {
            "dataset_size": size,
            "mean_seqeval": mean_seqeval_non_augmented,
            "sd_seqeval": sd_seqeval_non_augmented,
            "ci_lower": mean_seqeval_non_augmented - ci_non_aug,
            "ci_upper": mean_seqeval_non_augmented + ci_non_aug
        }
    )

# store the metrics
with open('/kaggle/working/metrics_datasetsize_eval.json', 'w') as f:
    json.dump(metrics_non_augmented, f)

Starting with size 4000
Current validation f1-score: 0.7344913151364764
Current validation f1-score: 0.7945900253592562
Current validation f1-score: 0.8063704945515507
Current validation f1-score: 0.8118323746918651
Current validation f1-score: 0.8026644462947544
Current validation f1-score: 0.7827557058326289
Current validation f1-score: 0.8205128205128205
Current validation f1-score: 0.831918505942275
Current validation f1-score: 0.8255813953488372
Current validation f1-score: 0.8055797733217088
Current validation f1-score: 0.8258283772302464
Best f1-score of the fold: 0.831918505942275
Current validation f1-score: 0.7267808836789901
Current validation f1-score: 0.795847750865052
Current validation f1-score: 0.7879325643300799
Current validation f1-score: 0.7923844061650045
Current validation f1-score: 0.8137082601054481
Current validation f1-score: 0.7996438112199465
Current validation f1-score: 0.7887817703768625
Current validation f1-score: 0.8089285714285716
Current validation f1

## Evaluation of Generative and Mixed Augmentations

In [9]:
# set a seed to ensure reproducibility
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# set number of folds and empty list to store metrics in
num_folds = 5
metrics = []

# set model name and hyperparameters
model_name = "roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 9e-06
weight_decay = 0.01
batch_size = 16
epochs = 10
tokenizer = AutoTokenizer.from_pretrained(model_name)
augmentation_ratio = 0.25

# create lists to store fold metrics in
fold_metrics_non_augmented = []
fold_metrics_gen_augmented = []
fold_metrics_mixed_augmented = []
    
# create K-Fold splits
kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop over each different fold
for fold, (train_idx, val_idx) in enumerate(kf.split(data_augmented)):
    
    # split the data according to the fold
    train_fold_data = [data_augmented[i] for i in train_idx]
    val_fold_data = [data_augmented[i] for i in val_idx]

    # create augmented training data (generative and mixed)
    augmented_train_data_generative = []
    augmented_train_data_mixed = []
    num_augmentations_to_add = int(len(train_fold_data) * augmentation_ratio)
    candidates_for_augmentation = random.sample(
        train_fold_data, k=min(num_augmentations_to_add, len(train_fold_data))
    )
    for original_item in candidates_for_augmentation:
        if "augmentations" in original_item and len(original_item["augmentations"]) > 0:
            aug_gen = original_item["augmentations"][-1]
            aug_mixed = random.choice([original_item["augmentations"][0], original_item["augmentations"][-1]])
            augmented_train_data_generative.append({
                "id": f"{original_item['id']}_aug_{aug_gen['method']}",
                "sentence": aug_gen["sentence"],
                "annotations": aug_gen["annotations"]
            })
            augmented_train_data_mixed.append({
                "id": f"{original_item['id']}_aug_{aug_mixed['method']}",
                "sentence": aug_mixed["sentence"],
                "annotations": aug_mixed["annotations"]
            })
    train_fold_data_gen_augmentations = train_fold_data + augmented_train_data_generative
    train_fold_data_mixed_augmentations = train_fold_data + augmented_train_data_mixed

    # create datasets and dataloaders
    train_dataset = TokenDataset(train_fold_data, tokenizer, tag_to_id, max_len=128)
    train_dataset_gen_augmentations = TokenDataset(train_fold_data_gen_augmentations, tokenizer, tag_to_id, max_len=128)
    train_dataset_mixed_augmentations = TokenDataset(train_fold_data_mixed_augmentations, tokenizer, tag_to_id, max_len=128)
    val_dataset = TokenDataset(val_fold_data, tokenizer, tag_to_id, max_len=128)
        
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    train_gen_augmentations_dataloader = DataLoader(train_dataset_gen_augmentations, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    train_mixed_augmentations_dataloader = DataLoader(train_dataset_mixed_augmentations, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

    # create model and optimizer for the non-augmentations
    model_non_augmented = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
    ).to(device)
    optimizer_non_aug = AdamW(model_non_augmented.parameters(), lr=lr, weight_decay=weight_decay)

    # loop through the epochs
    for epoch in range(epochs):
        model_non_augmented.train()
        total_loss = 0.0
        for batch in train_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tag_ids = batch["tag_ids"].to(device)

            optimizer_non_aug.zero_grad()

            with torch.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model_non_augmented(input_ids=input_ids, attention_mask=attention_mask, labels=tag_ids)
                loss = outputs.loss

            total_loss += loss.item()
            loss.backward()
            clip_grad_norm_(model_non_augmented.parameters(), 1.0)
            optimizer_non_aug.step()

    # evaluate on validation fold and store result
    all_true, all_pred, _ = run_testset_ner(
         model=model_non_augmented, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
    )
    metrics_seqeval = evaluate_seqeval(all_true, all_pred)
    val_f1 = metrics_seqeval["f1"]
    fold_metrics_non_augmented.append(val_f1)

    # create model and optimizer for the generative augmentation model
    model_gen_augmented = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
    ).to(device)
    optimizer_gen_aug = AdamW(model_gen_augmented.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(epochs):
        model_gen_augmented.train()
        total_loss = 0.0
        for batch in train_gen_augmentations_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tag_ids = batch["tag_ids"].to(device)

            optimizer_gen_aug.zero_grad()

            with torch.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model_gen_augmented(input_ids=input_ids, attention_mask=attention_mask, labels=tag_ids)
                loss = outputs.loss

            total_loss += loss.item()
            loss.backward()
            clip_grad_norm_(model_gen_augmented.parameters(), 1.0)
            optimizer_gen_aug.step()

    # evaluate on validation fold and store result
    all_true, all_pred, _ = run_testset_ner(
        model=model_gen_augmented, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
    )
    metrics_seqeval = evaluate_seqeval(all_true, all_pred)
    val_f1 = metrics_seqeval["f1"]
    fold_metrics_gen_augmented.append(val_f1)

    # create model and optimizer for the mixed augmentation model
    model_mixed_augmented = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
    ).to(device)
    optimizer_mixed_aug = AdamW(model_mixed_augmented.parameters(), lr=lr, weight_decay=weight_decay)


    for epoch in range(epochs):
        model_mixed_augmented.train()
        total_loss = 0.0
        for batch in train_mixed_augmentations_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tag_ids = batch["tag_ids"].to(device)

            optimizer_mixed_aug.zero_grad()

            with torch.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model_mixed_augmented(input_ids=input_ids, attention_mask=attention_mask, labels=tag_ids)
                loss = outputs.loss

            total_loss += loss.item()
            loss.backward()
            clip_grad_norm_(model_mixed_augmented.parameters(), 1.0)
            optimizer_mixed_aug.step()

    all_true, all_pred, _ = run_testset_ner(
        model=model_mixed_augmented, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
    )
    metrics_seqeval = evaluate_seqeval(all_true, all_pred)
    val_f1 = metrics_seqeval["f1"]
    fold_metrics_mixed_augmented.append(val_f1)

# take averages across folds
mean_seqeval_non_augmented = np.mean(fold_metrics_non_augmented)
mean_seqeval_gen_augmented = np.mean(fold_metrics_gen_augmented)
mean_seqeval_mixed_augmented = np.mean(fold_metrics_mixed_augmented)

# take standard deviation
sd_seqeval_non_augmented = np.std(fold_metrics_non_augmented)
sd_seqeval_gen_augmented = np.std(fold_metrics_gen_augmented)
sd_seqeval_mixed_augmented = np.std(fold_metrics_mixed_augmented)

# calculate confidence intervals
ci_non_aug = 1.96 * sd_seqeval_non_augmented / np.sqrt(5)
ci_gen_aug = 1.96 * sd_seqeval_gen_augmented / np.sqrt(5)
ci_mixed_aug = 1.96 * sd_seqeval_mixed_augmented / np.sqrt(5)

metrics.append(
    {
        "original_data": {
            "mean_seqeval": mean_seqeval_non_augmented,
            "sd_seqeval": sd_seqeval_non_augmented,
            "ci_lower": mean_seqeval_non_augmented - ci_non_aug,
            "ci_upper": mean_seqeval_non_augmented + ci_non_aug
        },
        "generative_augmentations": {
            "mean_seqeval": mean_seqeval_gen_augmented,
            "sd_seqeval": sd_seqeval_gen_augmented,
            "ci_lower": mean_seqeval_gen_augmented - ci_gen_aug,
            "ci_upper": mean_seqeval_gen_augmented + ci_gen_aug
        },
        "mixed_augmentations": {
            "mean_seqeval": mean_seqeval_mixed_augmented,
            "sd_seqeval": sd_seqeval_mixed_augmented,
            "ci_lower": mean_seqeval_mixed_augmented - ci_mixed_aug,
            "ci_upper": mean_seqeval_mixed_augmented + ci_mixed_aug
        }}
)

# store the metrics
with open('/kaggle/working/metrics_augmentations.json', 'w') as f:
    json.dump(metrics, f)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]